# Delete a policy

The harness has no delete feed. `DELETE /admin/policies/{id}` removes the row,
and the id disappears from `/ids`. The reconcile sweep (every 60 seconds)
hard-deletes every S3 version and checks the key is gone.
`kbs_snapshots_deleted_total` counts that delete when connector metrics answer.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo


## Precondition and create


In [ ]:
POLICY_ID = "pol-demo-delete"
KEY = f"raw/hr/policy/{POLICY_ID}.md"
BODY = "# Demo policy\n\nCreated by the notebook.\n"
PAYLOAD = {
    "policy_id": POLICY_ID,
    "title": "Demo policy",
    "category": "leave",
    "region": "DE",
    "format": "markdown",
    "native_acl": ["hr_employee"],
    "body": BODY,
}
hr_demo.delete_if_present("policy", POLICY_ID)
created = hr_demo.admin("POST", "/admin/policies", PAYLOAD)
print(created["policy_id"], "version", created["version"], created["native_acl"])


In [ ]:
def ready() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    return (
        BODY.encode() in hr_demo.object_body(KEY)
        and meta.get(hr_demo.META_ACL) == "hr_employee"
        and meta.get(hr_demo.META_VERSION) == str(created["version"])
        and len(meta.get(hr_demo.META_HASH, "")) == 64
        and all(meta.get(name) for name in hr_demo.META_KEYS)
    )

hr_demo.wait_until(ready, hr_demo.WRITE_WAIT, "create snapshot")
print(hr_demo.snapshot(KEY)["Metadata"])


## Delete


In [ ]:
deleted_before = hr_demo.metric("kbs_snapshots_deleted_total", source_id="hr")
hr_demo.admin("DELETE", f"/admin/policies/{POLICY_ID}")
missing = hr_demo.expect_status("GET", f"/policies/{POLICY_ID}", 404)
print("source", missing.status_code)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "delete sweep",
)
if deleted_before is None:
    print("connector metrics not reachable; S3 versions are the check")
else:

    def counted() -> bool:
        current = hr_demo.metric("kbs_snapshots_deleted_total", source_id="hr")
        return current is not None and current > deleted_before

    hr_demo.wait_until(counted, 30, "delete metric")
    deleted_after = hr_demo.metric("kbs_snapshots_deleted_total", source_id="hr")
    print("kbs_snapshots_deleted_total", deleted_before, "->", deleted_after)


## Cleanup


In [ ]:
hr_demo.delete_if_present("policy", POLICY_ID)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "cleanup delete",
)
print("removed", KEY)
